# PPO Portfolio Optimization: Paper Surprise, Robust Surprise, and Déjà Vu

This experiment compares alternative Surprise reward definitions using the exact
**real-data PPO control** from `FinRL_PortfolioOptimization_PPO_Synthetic_vs_Real`.

All variants use the same real data, per-ticker price scaling, 50-day EIIE
observation, `MultiInputPolicy`, PPO hyperparameters, **0.25% transaction cost**,
seeds, validation schedule, and external-reward backtest. The reward supplied to PPO
during training is the only experimental difference.


## Experimental protocol

The half-open periods and training settings match the Real-vs-Synthetic experiment:

| Block | Period | Purpose |
|---|---|---|
| Train | 2024-01-01 to 2024-10-01 | Fit the policy and real-data scaler |
| Validation | 2024-10-01 to 2025-01-01 | Select checkpoints using external reward |
| Test | 2025-01-01 to 2026-01-01 | Final untouched real-data comparison |

Only the original `with_fee=0.0025` condition is run. Formal mode therefore contains
`6 reward variants × 1 commission setting × 3 seeds = 18 runs`. Validation and every
reported financial metric use the original scaled log-return reward only. Intrinsic
reward is training-only.


In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
import glob
import os
import sys
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Image as NotebookImage
from stable_baselines3 import PPO
from tqdm.auto import tqdm


def _bootstrap_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "finrl").is_dir() and (candidate / "examples").is_dir():
            return candidate
    raise RuntimeError("Could not locate the FinRL project root.")


PROJECT_ROOT = _bootstrap_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from finrl.experiments.notebook_utils import check_calendar_coverage
from finrl.experiments.notebook_utils import environment_flag
from finrl.experiments.notebook_utils import read_real_ohlcv_csvs
from finrl.experiments.portfolio_intrinsic import evaluate_portfolio_periods
from finrl.experiments.portfolio_intrinsic import plan_portfolio_training
from finrl.experiments.portfolio_intrinsic import portfolio_environment_kwargs
from finrl.experiments.portfolio_intrinsic import portfolio_ppo_kwargs
from finrl.experiments.portfolio_intrinsic import prepare_real_portfolio_periods
from finrl.experiments.portfolio_intrinsic import resolve_variant_reward_config
from finrl.experiments.portfolio_intrinsic import train_portfolio_variant
from finrl.experiments.portfolio_intrinsic import VARIANT_WEIGHTS
from finrl.experiments.stock_intrinsic import plot_intrinsic_diagnostics
from finrl.experiments.stock_intrinsic import plot_learning_curves_by_variant
from finrl.experiments.stock_intrinsic import plot_period_equity_comparison
from finrl.experiments.stock_intrinsic import plot_sharpe_seed_selection
from finrl.experiments.stock_intrinsic import plot_test_metric_comparison
from finrl.experiments.stock_intrinsic import plot_test_portfolio_weights
from finrl.meta.env_portfolio_optimization.env_portfolio_optimization_gymnasium import (
    PortfolioOptimizationGymnasiumEnv,
)

warnings.filterwarnings("once")
DEVICE = os.getenv("FINRL_DEVICE", "cpu")
torch.set_num_threads(int(os.getenv("FINRL_TORCH_NUM_THREADS", "1")))
print("project root:", PROJECT_ROOT)
print("device:", DEVICE)


## 1. Configuration


In [ ]:
REAL_DATA_GLOB = os.getenv("FINRL_REAL_DATA_GLOB", "data/real/*.csv")
RESULT_ROOT = Path(
    os.getenv(
        "FINRL_INTRINSIC_RESULT_ROOT",
        str(PROJECT_ROOT / "results" / "ppo_portfolio_intrinsic_reward_ablation"),
    )
).resolve()

TRAIN_START, TRAIN_END = "2024-01-01", "2024-10-01"
VALIDATION_START, VALIDATION_END = "2024-10-01", "2025-01-01"
TEST_START, TEST_END = "2025-01-01", "2026-01-01"
EXPECTED_TICKERS = ["AAPL", "AMZN", "GOOGL", "MSFT", "NVDA"]
PRICE_COLUMNS = ["close", "high", "low"]
TIME_WINDOW = 50
INITIAL_AMOUNT = 100_000
REWARD_SCALING = 100.0

RUN_MODE = os.getenv("FINRL_INTRINSIC_RUN_MODE", "full").lower()
RUN_SETTINGS = {
    "smoke": {
        "seeds": [0, 1],
        "total_timesteps": 4_096,
        "eval_freq": 2_048,
    },
    "full": {
        "seeds": [0, 1, 2],
        "total_timesteps": 200_000,
        "eval_freq": 20_000,
    },
}
if RUN_MODE not in RUN_SETTINGS:
    raise ValueError(f"RUN_MODE must be one of {sorted(RUN_SETTINGS)}")

SEEDS = RUN_SETTINGS[RUN_MODE]["seeds"]
TOTAL_TIMESTEPS = RUN_SETTINGS[RUN_MODE]["total_timesteps"]
EVAL_FREQ = RUN_SETTINGS[RUN_MODE]["eval_freq"]
if os.getenv("FINRL_INTRINSIC_SEEDS"):
    SEEDS = [int(value) for value in os.environ["FINRL_INTRINSIC_SEEDS"].split(",")]
TOTAL_TIMESTEPS = int(
    os.getenv("FINRL_INTRINSIC_TOTAL_TIMESTEPS", str(TOTAL_TIMESTEPS))
)
EVAL_FREQ = int(os.getenv("FINRL_INTRINSIC_EVAL_FREQ", str(EVAL_FREQ)))
COMMISSION_CONFIGS = {"with_fee": 0.0025}
VARIANTS = list(VARIANT_WEIGHTS)
WARMUP_STEPS = int(os.getenv("FINRL_INTRINSIC_WARMUP_STEPS", "1024"))
REPLAY_CAPACITY = int(os.getenv("FINRL_INTRINSIC_REPLAY_CAPACITY", "100000"))
PPO_KWARGS = portfolio_ppo_kwargs(DEVICE)
for env_name, key in (
    ("FINRL_PPO_N_STEPS", "n_steps"),
    ("FINRL_PPO_BATCH_SIZE", "batch_size"),
    ("FINRL_PPO_N_EPOCHS", "n_epochs"),
):
    if os.getenv(env_name):
        PPO_KWARGS[key] = int(os.environ[env_name])

ARTIFACT_ROOT = RESULT_ROOT / RUN_MODE
TRAIN_CACHE_ROOT = RESULT_ROOT / "train_cache"
USE_TRAIN_CACHE = environment_flag("FINRL_USE_TRAIN_CACHE", True)
FORCE_RETRAIN = environment_flag("FINRL_FORCE_RETRAIN", False)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
TRAIN_CACHE_ROOT.mkdir(parents=True, exist_ok=True)

print("run mode:", RUN_MODE)
print("variants:", VARIANTS)
print("commission settings:", COMMISSION_CONFIGS)
print("seeds:", SEEDS)
print("timesteps per run:", f"{TOTAL_TIMESTEPS:,}")
print("validation frequency:", f"{EVAL_FREQ:,}")
print("formal run count:", len(VARIANTS) * len(COMMISSION_CONFIGS) * len(SEEDS))
print("training cache:", "enabled" if USE_TRAIN_CACHE else "disabled")
print("force retrain:", FORCE_RETRAIN)
print("output:", ARTIFACT_ROOT)


The common training reward is

\[
r_t=r_t^{ext}+\eta_t
\left(\alpha\hat r_t^{surprise}+\beta\hat r_t^{dejavu}\right).
\]

The six variants are:

| Variant | Surprise definition | Nominal α | β |
|---|---|---:|---:|
| baseline | none | 0 | 0 |
| paper_surprise | paper summed Gaussian NLL; no ReLU/RMS/bonus clipping | 0.05 | 0 |
| robust_surprise | positive running z-score of mean Gaussian NLL | 0.05 | 0 |
| dejavu | reconstruction MSE only | 0 | 0.05 |
| paper_surprise_dejavu | paper Surprise + Déjà vu | 0.05 | 0.05 |
| robust_surprise_dejavu | robust Surprise + Déjà vu | 0.05 | 0.05 |

For `paper_surprise` and `paper_surprise_dejavu`, the effective α is divided by observation dimension. With the
current `3 × 5 × 50 + 6 = 756` dimensional input, this gives `0.05 / 756`, preventing
the summed NLL from receiving a 756-fold reward-scale advantage. This changes only an
unspecified hyperparameter scale; the Surprise score itself remains formula-faithful.

Every episode starts fully in cash. The policy outputs six continuous actions—cash
plus five stocks—which the unchanged parent environment scales and softmaxes into
portfolio weights.


## 2. Load, validate, split, and scale real data


In [ ]:
real_files = [
    Path(path) for path in sorted(glob.glob(str(PROJECT_ROOT / REAL_DATA_GLOB)))
]
real_ohlcv = read_real_ohlcv_csvs(real_files, EXPECTED_TICKERS)
check_calendar_coverage(real_ohlcv, TRAIN_START, VALIDATION_END, "real train+validation")
check_calendar_coverage(real_ohlcv, TEST_START, TEST_END, "real test")

periods, real_scaler = prepare_real_portfolio_periods(
    real_ohlcv,
    EXPECTED_TICKERS,
    TRAIN_START,
    TRAIN_END,
    VALIDATION_START,
    VALIDATION_END,
    TEST_START,
    TEST_END,
    time_window=TIME_WINDOW,
    price_columns=PRICE_COLUMNS,
)
train = periods["train"]
validation = periods["validation"]
test = periods["test"]
ENV_KWARGS_BY_COMMISSION = {
    name: portfolio_environment_kwargs(
        commission,
        cwd=PROJECT_ROOT,
        initial_amount=INITIAL_AMOUNT,
        time_window=TIME_WINDOW,
        reward_scaling=REWARD_SCALING,
        price_columns=PRICE_COLUMNS,
    )
    for name, commission in COMMISSION_CONFIGS.items()
}

analysis_starts = {
    "train": pd.Timestamp(TRAIN_START),
    "validation": pd.Timestamp(VALIDATION_START),
    "test": pd.Timestamp(TEST_START),
}
period_summary = pd.DataFrame(
    [
        {
            "period": label,
            "input_first_date": frame["date"].min(),
            "analysis_first_date": analysis_starts[label],
            "last_date": frame["date"].max(),
            "input_dates": frame["date"].nunique(),
            "analysis_dates": frame.loc[
                frame["date"] >= analysis_starts[label], "date"
            ].nunique(),
            "tickers": frame["tic"].nunique(),
        }
        for label, frame in periods.items()
    ]
)
print("per-ticker train scales:", real_scaler.scales)
period_summary


### Resolved reward-variant configuration


In [ ]:
variant_configuration = pd.DataFrame(
    [
        {
            "variant": variant,
            "surprise_mode": resolved["surprise_mode"],
            "observation_dim": resolved["observation_dim"],
            "nominal_alpha": resolved["nominal_alpha"],
            "effective_alpha": resolved["effective_alpha"],
            "beta": resolved["beta"],
        }
        for variant in VARIANTS
        for resolved in [
            resolve_variant_reward_config(
                train,
                variant,
                ENV_KWARGS_BY_COMMISSION["with_fee"],
            )
        ]
    ]
)
variant_configuration.to_csv(
    ARTIFACT_ROOT / "reward_variant_configuration.csv", index=False
)
variant_configuration


### Minimal real-control sanity check


In [ ]:
sanity_env = PortfolioOptimizationGymnasiumEnv(
    train,
    **ENV_KWARGS_BY_COMMISSION["with_fee"],
)
sanity_observation, _ = sanity_env.reset(seed=SEEDS[0])
print("state:", sanity_observation["state"].shape)
print("last action:", sanity_observation["last_action"])
print("action space:", sanity_env.action_space)
print("max achievable single weight:", f"{sanity_env.max_achievable_weight():.6f}")
assert sanity_observation["state"].shape == (
    len(PRICE_COLUMNS),
    len(EXPECTED_TICKERS),
    TIME_WINDOW,
)
assert sanity_observation["last_action"].shape == (len(EXPECTED_TICKERS) + 1,)
assert sanity_env.action_space.shape == (len(EXPECTED_TICKERS) + 1,)


## 3. Training process

The preflight fingerprints every semantic run before building a PPO environment.
Complete cache hits load first; only misses train. The outer bar tracks runs and the
inner bar tracks PPO timesteps. At timestep 0 and every `EVAL_FREQ`, deterministic
external reward is measured on real train and real validation. Only real-validation
external reward selects the checkpoint. All runs use `with_fee=0.0025`. Each run's
UTC timestamps and wall-clock duration are persisted immediately to `run_timings.csv`.


In [ ]:
run_specs = [
    {
        "variant": variant,
        "commission_name": commission_name,
        "commission": commission,
        "seed": seed,
    }
    for seed in SEEDS
    for commission_name, commission in COMMISSION_CONFIGS.items()
    for variant in VARIANTS
]
run_plans = [
    plan_portfolio_training(
        train=train,
        validation=validation,
        variant=spec["variant"],
        commission_name=spec["commission_name"],
        commission=spec["commission"],
        total_timesteps=TOTAL_TIMESTEPS,
        eval_freq=EVAL_FREQ,
        seed=spec["seed"],
        warmup_steps=WARMUP_STEPS,
        env_kwargs=ENV_KWARGS_BY_COMMISSION[spec["commission_name"]],
        ppo_kwargs=PPO_KWARGS,
        cache_dir=TRAIN_CACHE_ROOT,
        use_cache=USE_TRAIN_CACHE,
        force_retrain=FORCE_RETRAIN,
        replay_capacity=REPLAY_CAPACITY,
    )
    for spec in tqdm(run_specs, desc="Checking training cache", unit="run")
]
cache_hit_plans = [plan for plan in run_plans if plan.cache_hit]
cache_miss_plans = [plan for plan in run_plans if not plan.cache_hit]
print(
    "cache preflight:",
    f"{len(cache_hit_plans)} hit(s), {len(cache_miss_plans)} miss(es)",
)


def train_from_plan(plan):
    return train_portfolio_variant(
        train=train,
        validation=validation,
        plan=plan,
        total_timesteps=TOTAL_TIMESTEPS,
        eval_freq=EVAL_FREQ,
        warmup_steps=WARMUP_STEPS,
        env_kwargs=ENV_KWARGS_BY_COMMISSION[plan.commission_name],
        ppo_kwargs=PPO_KWARGS,
        show_progress=True,
        replay_capacity=REPLAY_CAPACITY,
    )


RUN_TIMINGS_PATH = ARTIFACT_ROOT / "run_timings.csv"
timing_records = []


def execute_timed_run(plan):
    started_at = datetime.now(timezone.utc)
    started_clock = time.perf_counter()
    status = "failed"
    error_type = ""
    try:
        run = train_from_plan(plan)
        status = "completed"
        return run
    except Exception as error:
        error_type = type(error).__name__
        raise
    finally:
        finished_at = datetime.now(timezone.utc)
        timing_records.append(
            {
                "execution_order": len(timing_records) + 1,
                "run_mode": RUN_MODE,
                "run_id": (
                    f"{plan.variant}__{plan.commission_name}__seed_{plan.seed}"
                ),
                "variant": plan.variant,
                "commission_name": plan.commission_name,
                "seed": plan.seed,
                "cache_hit": plan.cache_hit,
                "status": status,
                "error_type": error_type,
                "started_at_utc": started_at.isoformat(),
                "finished_at_utc": finished_at.isoformat(),
                "elapsed_seconds": round(
                    time.perf_counter() - started_clock, 6
                ),
            }
        )
        pd.DataFrame(timing_records).to_csv(RUN_TIMINGS_PATH, index=False)


runs_by_key = {}
for plan in tqdm(cache_hit_plans, desc="Loading cache hits", unit="run"):
    key = (plan.variant, plan.commission_name, plan.seed)
    runs_by_key[key] = execute_timed_run(plan)

if cache_miss_plans:
    miss_progress = tqdm(
        cache_miss_plans,
        desc="Training cache misses",
        unit="run",
        position=0,
    )
    for plan in miss_progress:
        miss_progress.set_postfix_str(
            f"{plan.variant} | {plan.commission_name} | seed {plan.seed}",
            refresh=True,
        )
        key = (plan.variant, plan.commission_name, plan.seed)
        runs_by_key[key] = execute_timed_run(plan)
else:
    print("No cache misses to train.")

runs = [
    runs_by_key[(plan.variant, plan.commission_name, plan.seed)]
    for plan in run_plans
]
run_table = pd.DataFrame(
    [
        {
            "variant": run.variant,
            "commission": run.commission_name,
            "seed": run.seed,
            "cache_hit": run.cache_hit,
            "cache_id": run.cache_key[:12],
            "best_timestep": run.best_timestep,
            "best_real_validation_reward": (
                run.validation_log["real_valid_reward"].max()
                if not run.validation_log.empty
                else np.nan
            ),
        }
        for run in runs
    ]
)
run_table.to_csv(ARTIFACT_ROOT / "training_runs.csv", index=False)
print("training cache hits:", f"{int(run_table['cache_hit'].sum())}/{len(run_table)}")
print("saved run timings:", RUN_TIMINGS_PATH)
run_table


## 4. Learning curves

Each figure contains one reward variant aggregated across the same seeds as the
original experiment. Bands show ±1 SD. Deterministic train and validation curves
always use external financial reward; sampled-policy reward is the reward PPO actually
received during training. All figures use the 0.25% fee condition.


In [ ]:
learning_paths = {}
for commission_name in COMMISSION_CONFIGS:
    commission_runs = [
        run for run in runs if run.commission_name == commission_name
    ]
    figures = plot_learning_curves_by_variant(commission_runs)
    for variant, figure in figures.items():
        path = ARTIFACT_ROOT / (
            f"{commission_name}__{variant}__learning_curves_all_seeds.png"
        )
        figure.savefig(path, dpi=160, bbox_inches="tight")
        plt.close(figure)
        learning_paths[(commission_name, variant)] = path
learning_paths


In [ ]:
NotebookImage(filename=str(learning_paths[("with_fee", "baseline")]))


In [ ]:
NotebookImage(filename=str(learning_paths[("with_fee", "paper_surprise")]))


In [ ]:
NotebookImage(filename=str(learning_paths[("with_fee", "robust_surprise")]))


In [ ]:
NotebookImage(filename=str(learning_paths[("with_fee", "dejavu")]))


In [ ]:
NotebookImage(filename=str(learning_paths[("with_fee", "paper_surprise_dejavu")]))


In [ ]:
NotebookImage(filename=str(learning_paths[("with_fee", "robust_surprise_dejavu")]))


### Intrinsic-reward diagnostics (additional)

These panels compare paper summed NLL, robust centered-z Surprise, Déjà vu, and both
Surprise + Déjà vu combinations. A large or negative paper score is
an experimental result rather than a plotting error; its effective α is dimension
adjusted before reward composition.


In [ ]:
intrinsic_log_frames = []
for run in runs:
    if run.variant == "baseline" or run.intrinsic_log.empty:
        continue
    frame = run.intrinsic_log.copy()
    frame["variant"] = run.variant
    frame["commission_name"] = run.commission_name
    frame["seed"] = run.seed
    intrinsic_log_frames.append(frame)
intrinsic_log = pd.concat(intrinsic_log_frames, ignore_index=True)

intrinsic_paths = {}
intrinsic_summaries = []
for commission_name in COMMISSION_CONFIGS:
    commission_log = intrinsic_log.loc[
        intrinsic_log["commission_name"] == commission_name
    ]
    figure, curve_summary, model_summary = plot_intrinsic_diagnostics(
        commission_log
    )
    path = ARTIFACT_ROOT / f"{commission_name}__intrinsic_reward_diagnostics.png"
    figure.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(figure)
    intrinsic_paths[commission_name] = path
    model_summary.insert(1, "commission_name", commission_name)
    intrinsic_summaries.append(model_summary)
pd.concat(intrinsic_summaries, ignore_index=True).to_csv(
    ARTIFACT_ROOT / "intrinsic_reward_summary.csv", index=False
)
intrinsic_paths


In [ ]:
NotebookImage(filename=str(intrinsic_paths["with_fee"]))


## 5. External-reward backtests and uniform benchmark

Every saved PPO checkpoint is evaluated deterministically on train, validation, and
test using the unmodified portfolio environment with a 0.25% fee. The benchmark uses
the same uniform portfolio action as the Real-vs-Synthetic notebook. No intrinsic
reward or novelty-model update occurs here.


In [ ]:
(
    ppo_period_summary,
    period_results,
    benchmark_summary,
    benchmark_results,
) = evaluate_portfolio_periods(
    periods,
    runs,
    cwd=PROJECT_ROOT,
    initial_amount=INITIAL_AMOUNT,
    time_window=TIME_WINDOW,
    reward_scaling=REWARD_SCALING,
)
backtest_summary = pd.concat(
    [ppo_period_summary, benchmark_summary], ignore_index=True
)
ppo_period_summary.to_csv(ARTIFACT_ROOT / "ppo_period_metrics.csv", index=False)
benchmark_summary.to_csv(ARTIFACT_ROOT / "benchmark_metrics.csv", index=False)
backtest_summary.to_csv(ARTIFACT_ROOT / "backtest_results.csv", index=False)
backtest_summary


## 6. Backtest Sharpe and test metrics

The Sharpe plot retains every seed and marks the seed nearest each variant's median
test Sharpe. The metric plot shows all-seed mean ± SD and the deterministic uniform
benchmark, all under the 0.25% fee condition.


In [ ]:
sharpe_paths = {}
metric_paths = {}
test_model_summaries = []
for commission_name in COMMISSION_CONFIGS:
    test_summary = ppo_period_summary.loc[
        (ppo_period_summary["period"] == "test")
        & (ppo_period_summary["commission_name"] == commission_name)
    ].reset_index(drop=True)
    test_benchmark = benchmark_summary.loc[
        (benchmark_summary["period"] == "test")
        & (benchmark_summary["commission_name"] == commission_name)
    ].iloc[0]

    sharpe_figure, representatives = plot_sharpe_seed_selection(test_summary)
    sharpe_path = ARTIFACT_ROOT / f"{commission_name}__median_sharpe_seed_selection.png"
    sharpe_figure.savefig(sharpe_path, dpi=160, bbox_inches="tight")
    plt.close(sharpe_figure)
    representatives.to_csv(
        ARTIFACT_ROOT / f"{commission_name}__median_sharpe_representatives.csv",
        index=False,
    )
    sharpe_paths[commission_name] = sharpe_path

    metrics_figure, model_summary = plot_test_metric_comparison(
        test_summary, test_benchmark
    )
    metrics_path = ARTIFACT_ROOT / f"{commission_name}__test_metrics_vs_benchmark.png"
    metrics_figure.savefig(metrics_path, dpi=160, bbox_inches="tight")
    plt.close(metrics_figure)
    model_summary.insert(1, "commission_name", commission_name)
    test_model_summaries.append(model_summary)
    metric_paths[commission_name] = metrics_path

test_model_summary = pd.concat(test_model_summaries, ignore_index=True)
test_model_summary.to_csv(
    ARTIFACT_ROOT / "test_metrics_aggregated.csv", index=False
)


In [ ]:
NotebookImage(filename=str(sharpe_paths["with_fee"]))


In [ ]:
NotebookImage(filename=str(metric_paths["with_fee"]))


## 7. Train, validation, and test equity curves

All plots use Growth of $1 under `with_fee=0.0025`. Reward variants are aggregated
across seeds as mean ± SD; the uniform benchmark is the dashed black line. Tables
report the exact ranking for the corresponding period.


In [ ]:
equity_paths = {}
period_titles = {
    "train": f"Real train [{TRAIN_START}, {TRAIN_END})",
    "validation": f"Real validation [{VALIDATION_START}, {VALIDATION_END})",
    "test": f"Real test [{TEST_START}, {TEST_END})",
}
for commission_name in COMMISSION_CONFIGS:
    for period in ("train", "validation", "test"):
        figure, equity, ranking = plot_period_equity_comparison(
            period,
            period_results[commission_name][period],
            benchmark_results[commission_name][period],
            f"{period_titles[period]} — {commission_name}",
        )
        path = ARTIFACT_ROOT / (
            f"{commission_name}__ppo_vs_benchmark_{period}_equity_curves.png"
        )
        figure.savefig(path, dpi=160, bbox_inches="tight")
        plt.close(figure)
        equity.to_csv(
            ARTIFACT_ROOT / f"{commission_name}__{period}_equity_curves.csv",
            index=False,
        )
        ranking.to_csv(
            ARTIFACT_ROOT / f"{commission_name}__{period}_equity_ranking.csv",
            index=False,
        )
        equity_paths[(commission_name, period)] = path
equity_paths


In [ ]:
NotebookImage(filename=str(equity_paths[("with_fee", "train")]))


In [ ]:
NotebookImage(filename=str(equity_paths[("with_fee", "validation")]))


In [ ]:
NotebookImage(filename=str(equity_paths[("with_fee", "test")]))


## 8. Portfolio weights

The stacked areas show chosen cash and stock weights through the real test period,
averaged across seeds for each reward variant under the 0.25% fee condition. The
table reports exact mean weights.


In [ ]:
weight_paths = {}
weight_summaries = []
for commission_name in COMMISSION_CONFIGS:
    figure, mean_weights = plot_test_portfolio_weights(
        period_results[commission_name]["test"]
    )
    path = ARTIFACT_ROOT / f"{commission_name}__real_test_portfolio_weights.png"
    figure.savefig(path, dpi=160, bbox_inches="tight")
    plt.close(figure)
    mean_weights.insert(1, "commission_name", commission_name)
    weight_summaries.append(mean_weights)
    weight_paths[commission_name] = path
pd.concat(weight_summaries, ignore_index=True).to_csv(
    ARTIFACT_ROOT / "mean_test_portfolio_weights.csv", index=False
)
weight_paths


In [ ]:
NotebookImage(filename=str(weight_paths["with_fee"]))


## 9. Interpretation checklist

- Compare all variants under the single `with_fee=0.0025` condition.
- Treat `paper_surprise` as the paper summed-NLL implementation, not the robust default.
- Negative paper Surprise rewards or scale spikes are valid stability findings.
- Compare differences with cross-seed bands, not only point estimates.
- Intrinsic diagnostics explain the mechanism; they are not financial P&L.
- Backtest, Sharpe, equity, and portfolio-weight figures use external outcomes only.
- Baseline uses the same real-data PortfolioOptimization + EIIE PPO control; only the
  training reward differs.


## Next steps

Run smoke mode first, then the formal 18-run experiment. The complete intrinsic
controller checkpoint includes its replay pool and can be large. The `paper_surprise`
variant should always receive a smoke check before a long run because summed Gaussian
NLL may be negative or exhibit large spikes.

```bash
FINRL_INTRINSIC_RUN_MODE=smoke \
FINRL_USE_TRAIN_CACHE=1 \
FINRL_DEVICE=cpu \
jupyter nbconvert --to notebook --execute --inplace \
  examples/FinRL_PortfolioOptimization_PPO_IntrinsicReward_Ablation.ipynb \
  --ExecutePreprocessor.timeout=-1
```

For the formal `6 variants × 3 seeds` experiment, change
`FINRL_INTRINSIC_RUN_MODE=full`.


In [ ]:
best_rows = test_model_summary.loc[
    test_model_summary.groupby("commission_name")["sharpe_mean"].idxmax()
]
print("Highest mean PPO Sharpe by commission:")
for _, row in best_rows.iterrows():
    print(
        f"  {row['commission_name']}: {row['variant']} ",
        f"n={int(row['seed_count'])}, ",
        f"Sharpe={row['sharpe_mean']:.3f} ± {row['sharpe_std']:.3f}",
        sep="",
    )
print("Training cache hits:", f"{int(run_table['cache_hit'].sum())}/{len(run_table)}")
print("Treat differences smaller than cross-seed variability cautiously.")
